In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("homework") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.NullPointerException
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:559)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
	at sun.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at sun.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
	at sun.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
	at java.lang.reflect.Constructor.newInstance(Constructor.java:423)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:357)
	at py4j.Gateway.invoke(Gateway.java:238)
	at py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
	at py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
	at py4j.GatewayConnection.run(GatewayConnectio

In [ ]:
from pyspark.sql import SparkSession
import socket

print("=" * 60)
print("ПОДКЛЮЧЕНИЕ К SPARK КЛАСТЕРУ")
print("=" * 60)

# Используем точный IP из диагностики
master_ip = "172.18.0.3"
master_url = f"spark://{master_ip}:7077"
hostname = socket.gethostname()

print(f"IP мастера: {master_ip}")
print(f"Хост Jupyter: {hostname}")

# Создаем SparkSession с правильной конфигурацией
spark = (SparkSession.builder
    .appName("homework")
    .master(master_url)
    # Отключаем UI для избежания конфликтов
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    # Настройки драйвера
    .config("spark.driver.host", hostname)
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "0")  # Случайный порт
    .config("spark.driver.blockManager.port", "0")
    # Настройки исполнителей
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "2")
    # Таймауты
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
    .getOrCreate())

print("✅ SparkSession успешно создана!")
print(f"Версия Spark: {spark.version}")
print(f"Master URL: {spark.sparkContext.master}")

# Проверка работы
print("\nТестовый DataFrame:")
df = spark.range(10)
df.show()

# Проверка работы с HDFS
print("\nПроверка HDFS:")
try:
    # Пробуем прочитать корневую директорию HDFS
    hdfs_path = "hdfs://namenode:9000/"
    test_df = spark.createDataFrame([(1, "test")], ["id", "value"])
    test_df.write.mode("overwrite").parquet(hdfs_path + "test_output")
    print("✅ Запись в HDFS успешна!")
    
    read_df = spark.read.parquet(hdfs_path + "test_output")
    print("✅ Чтение из HDFS успешно!")
    read_df.show()
except Exception as e:
    print(f"⚠️ HDFS проверка: {e}")

ПОДКЛЮЧЕНИЕ К SPARK КЛАСТЕРУ
IP мастера: 172.18.0.3
Хост Jupyter: 25abf1f4fd0f


In [2]:
spark.stop()

NameError: name 'spark' is not defined